In [2]:
import json
import io
import pandas as pd
import sys, os
from rels_utils import *

In [36]:
neg = 0
pos = 0
zero = 0
with open('/home/garrett/work/coef_from_word_tarballs/checkpoint_6loop/htcondor/dumped/debug/170962994/eval.final27.boots.55', 'r') as symbfile:
    raw_evals = json.load(symbfile)
    with open('../ChromoBoot_data/processed_data/coef_from_word/relations_6loop/rel_instances_final27.json', 'r') as openfile:
        truth_symb = json.load(openfile)
        raw_evals = replace_trivial0_terms(raw_evals, return_symb=True)
        rel_instance_eval_symb = update_rel_instances_in_symb(truth_symb, raw_evals)
        count = 0
        for inst_truth,inst_eval in zip(truth_symb,rel_instance_eval_symb):
            isBad=False
            for key in inst_truth.keys():
                if inst_eval[key][0] is None: continue
                if int(inst_truth[key][0]) != int(inst_eval[key][0]):
                    isBad=True
                    count += 1
            if not isBad:
                print(inst_truth)
                print(inst_eval)
                for i,key in enumerate(inst_truth.keys()):
                    if (inst_truth[key][0] != 0): print(i)
                print()


{'afbbaaaecdbd': [128, 1], 'afbbaaaebfff': [0, -0.5], 'afbbaaaedcee': [0, 0.5], 'afbbaaaeecee': [464, -0.5], 'afbbaaaecddd': [208, 0.5], 'afbbaaaedbdd': [0, 0.5], 'afbbaaaefbdd': [0, -0.5]}
{'afbbaaaecdbd': [128, 1], 'afbbaaaebfff': [0, -0.5], 'afbbaaaedcee': [0, 0.5], 'afbbaaaeecee': [464, -0.5], 'afbbaaaecddd': [208, 0.5], 'afbbaaaedbdd': [0, 0.5], 'afbbaaaefbdd': [0, -0.5]}
0
3
4

{'ccabffbccdbd': [384, 1], 'ccabffbcbfff': [768, -0.5], 'ccabffbcdcee': [960, 0.5], 'ccabffbcecee': [992, -0.5], 'ccabffbccddd': [0, 0.5], 'ccabffbcdbdd': [32, 0.5], 'ccabffbcfbdd': [0, -0.5]}
{'ccabffbccdbd': [384, 1], 'ccabffbcbfff': [768, -0.5], 'ccabffbcdcee': [960, 0.5], 'ccabffbcecee': [992, -0.5], 'ccabffbccddd': [0, 0.5], 'ccabffbcdbdd': [32, 0.5], 'ccabffbcfbdd': [0, -0.5]}
0
1
2
3
5

{'bffbceaecdbd': [1024, 1], 'bffbceaebfff': [0, -0.5], 'bffbceaedcee': [0, 0.5], 'bffbceaeecee': [2752, -0.5], 'bffbceaecddd': [704, 0.5], 'bffbceaedbdd': [0, 0.5], 'bffbceaefbdd': [0, -0.5]}
{'bffbceaecdbd': [1024, 

In [93]:
def make_4loop(min_length=6, max_length=6, seed=0
):
    #vocab = ['+', '-', 'a', 'b', 'c', 'd', 'e', 'f','g','h'] +[f"v{i:02d}" for i in range(93)]
    path = "../ChromoBoot_data/processed_data/coef_from_word/5loop/loop5.data"
    sents, tags = [], []
    with io.open(path, mode="r", encoding="utf-8") as f:
        # either reload the entire file, or the first N lines
        # (for the training set)
        lines = [line.rstrip().split("|") for line in f]
        data = [xy.split("\t") for _, xy in lines]
        data = [xy for xy in data if len(xy) == 2]
        for xy in data:
            word = ''.join(str(x) for x in xy[0].split(" "))
            sents.append(word)
            mytag = ''.join(str(x) for x in xy[1].split(" "))
            tags.append(mytag)
    return pd.DataFrame({"sent": sents, "tags": tags})


def numletters(df):
    chars=0
    charslist=[]
    for char in df["sent"]:
        if char not in charslist:
            charslist.append(char)
            chars += 1
    return chars

def word_to_binary(string):
    result=[]
    for char in string:
        if char in ['a','b','c']: result.append(0)
        if char in ['d','e','f']: result.append(1)
    return ''.join([str(x) for x in result])

def word_to_runs(string):
    result=[]
    this_run=1
    for i in range(len(string)):
        if i==0:
            continue
            
        if string[i]==string[i-1]:
            this_run += 1
            #continue run  
        else:
            result.append('[')
            if string[i-1] in ['d','e','f']: result.append('-')
            result.append(this_run)
            result.append(']')
            this_run=1

        if i==len(string)-1:
            result.append('[')
            if string[i] in ['d','e','f']: result.append('-')
            result.append(this_run)
            result.append(']')        
            

    return ''.join([str(x) for x in result])
        

In [118]:
mydf = make_4loop()
#pd.set_option('display.max_rows', None)
pd.set_option('display.min_rows', 500)
mydf['mag']=np.abs(mydf.tags.astype(int))
mydf["binary"]=mydf['sent'].apply(word_to_binary)
mydf["runs"]=mydf['sent'].apply(word_to_runs)



coefs = mydf['tags'].unique().tolist()
for coef in coefs:
    if abs(int(coef)) not in dummylist:
        dummylist.append(abs(int(coef)))
    else:
        print(abs(int(coef)))

32
24
48
48
60
36
32
468
24
600
476
8
560
8
16
52
64
168
480
40
68
576
80
80
156
40
480
72
104
240
96
112
288
336
168
128
120
96
104
800
64
152
124
136
152
88
832
88
56
52
68
400
16
224
112
224
72
144
144
2432
2432
92
160
36
44
120
176
56
100
44
20
516
100
328
760
208
408
368
232
640
256
192
240
336
976
288
464
864
960
76
76
28
28
192
352
184
184
528
136
512
84
164
132
960
272
848
784
848
1200
400
128
2208
156
352
608
744
544
296
264
116
832
728
864
976
18
344
148
320
880
132
208
304
60
160
416
1568
256
728
1472
4160
664
4352
432
744
1600
232
272
688
384
440
12
440
528
1504
800
448
592
176
496
248
560
936
148
816
992
1248
416
18
752
1760
464
576
2304
720
5760
1920
304
248
1664
512
4480
768
448
784
1272
640
188
20
540
936
544
372
188
320
11520
5248
5376
1184
1792
2144
124
1152
1536
372
3520
3552
5568
5728
7760
736
368
672
2176
496
648
1216
2712
3872
1344
5376
1184
1712
616
5280
312
200
1248
688
3264
736
1888
1392
1424
1344
8640
2464
5120
1504
1200
12
584
376
768
672
952
384
1088
4976
23

In [277]:
mydf[mydf["sent"]=="bdcacbdddd"] 

,sent,tags,mag,binary,runs
200444,bdbccbdddd,+224,224,0100001111,[1][-1][1][2][1][-4]
